# pyserial 学习笔记（入门速通）

> 目标：掌握串口发现、打开配置、读写通信、超时控制与常见异常处理。

pyserial 是 Python 串口通信的事实标准库。注意包名与导入名的区别：**安装时用 `pip install pyserial`，代码中写 `import serial`**，而不是 `import pyserial`[citation:12][citation:15]。它支持 Windows、Linux、macOS、BSD，提供统一的 `Serial` 类接口，屏蔽了不同操作系统底层 API 的差异[citation:7]。

**环境准备**
```bash
pip install pyserial
```

> 本 Notebook 中涉及实际串口设备的代码需要连接硬件才能运行。没有硬件时，可用 `loop://` 虚拟串口做基础验证。

In [1]:
import serial
import serial.tools.list_ports
print("pyserial version:", serial.__version__)

pyserial version: 3.5


## 1. 发现可用串口

在打开串口之前，先确认系统识别到了哪些设备。`serial.tools.list_ports.comports()` 返回一个 `ListPortInfo` 对象列表，每个对象包含端口的关键信息[citation:1][citation:3]。

| 属性 | 含义 | 示例值 |
|---|---|---|
| `device` | 设备路径 / 端口名 | `/dev/ttyUSB0`、`COM3` |
| `description` | 人类可读的描述 | `USB-SERIAL CH340` |
| `hwid` | 硬件 ID | `USB VID:PID=1A86:7523` |
| `vid` / `pid` | USB 厂商/产品 ID | 整数或 `None` |

**命令行方式**：`python -m serial.tools.list_ports` 可直接在终端列出端口，支持传入正则表达式过滤[citation:1][citation:2]。

**注意**：枚举结果在部分平台上可能不完整或缺少详细描述，不能完全依赖它判断端口是否存在[citation:1]。

In [2]:
ports = serial.tools.list_ports.comports()
print(f"发现 {len(ports)} 个串口设备:\n")
for p in ports:
    print(f"  设备: {p.device}")
    print(f"  描述: {p.description}")
    print(f"  硬件ID: {p.hwid}\n")

if not ports:
    print("未检测到串口设备（无硬件时属正常）")

发现 4 个串口设备:

  设备: COM6
  描述: 蓝牙链接上的标准串行 (COM6)
  硬件ID: BTHENUM\{00001101-0000-1000-8000-00805F9B34FB}_VID&000105D6_PID&000A\8&2E55756A&0&08F0B609757B_C00000000

  设备: COM15
  描述: 蓝牙链接上的标准串行 (COM15)
  硬件ID: BTHENUM\{00001101-0000-1000-8000-00805F9B34FB}_LOCALMFG&0002\8&2E55756A&0&4142FFA8E530_C00000000

  设备: COM7
  描述: 蓝牙链接上的标准串行 (COM7)
  硬件ID: BTHENUM\{00001101-0000-1000-8000-00805F9B34FB}_LOCALMFG&0000\8&2E55756A&0&000000000000_00000000

  设备: COM16
  描述: 蓝牙链接上的标准串行 (COM16)
  硬件ID: BTHENUM\{00001101-0000-1000-8000-00805F9B34FB}_LOCALMFG&0000\8&2E55756A&0&000000000000_00000001



## 2. 打开串口

`serial.Serial()` 是最常用的入口。最简形式只需端口名，其余参数使用默认值：**波特率 9600、8 数据位、无校验、1 停止位、无超时**[citation:1]。

**关键参数**

| 参数 | 含义 | 常用值 |
|---|---|---|
| `port` | 端口名 | `'COM3'`、`'/dev/ttyUSB0'` |
| `baudrate` | 波特率 | 9600、115200 等，**必须与设备端一致** |
| `bytesize` | 数据位 | `serial.EIGHTBITS`（默认） |
| `parity` | 校验位 | `serial.PARITY_NONE`（默认）、`EVEN`、`ODD` |
| `stopbits` | 停止位 | `serial.STOPBITS_ONE`（默认）、`TWO` |
| `timeout` | 读超时（秒） | **强烈建议设置**，`None` 会永久阻塞 |
| `write_timeout` | 写超时（秒） | 默认 `None`（阻塞） |
| `rtscts` | 硬件流控 | `True` / `False` |
| `xonxoff` | 软件流控 | `True` / `False` |

**两种打开方式**
1. 构造时直接传入所有参数并打开（最常用）。
2. 先创建空实例，再逐个设置属性，最后调用 `open()`——适合配置来自外部参数的场景[citation:1]。

**上下文管理器**：`with serial.Serial(...) as ser:` 在退出时自动关闭端口，推荐使用[citation:1]。

**易错点**
- 波特率不匹配会导致乱码或无响应。
- 不设 `timeout` 时，`read()` 在无数据时会**永远阻塞**。
- Windows 上 COM 端口号超过 COM9 时，需写成 `'\\\\.\\COM10'` 的转义形式[citation:6]。

In [ ]:
# 方式一：构造时打开（有硬件时替换端口名）
# ser = serial.Serial('/dev/ttyUSB0', 115200, timeout=1)

# 方式二：延迟配置
ser = serial.Serial()
ser.baudrate = 19200
ser.port = 'COM1'          # 示例；实际使用前先确认端口存在
ser.timeout = 1
print("配置完成（未打开）:", ser)

# 上下文管理器（推荐）：退出 with 块自动 close
# with serial.Serial('COM3', 9600, timeout=1) as ser:
#     ser.write(b'hello')

print("\n提示：以上 open() 需要真实设备，此处仅演示配置流程")

## 3. 读写操作

pyserial 提供文件式 API：`read`、`write`、`readline` 等方法与操作文件类似[citation:7]。

**写入**
- `write(data)`：**只接受 bytes**，Python 3 中字符串需先 `.encode()`[citation:3][citation:5]。
- 返回实际写入的字节数。

**读取**

| 方法 | 行为 | 注意 |
|---|---|---|
| `read(size=1)` | 读取指定字节数 | 数据不足时会等待直到超时，返回实际读到的部分 |
| `readline()` | 读到换行符 `\n` | **必须设置 `timeout`**，否则无换行时永久阻塞[citation:1][citation:2] |
| `readlines()` | 读所有行 | 依赖 `timeout` 作为 EOF 判断，行为在串口场景下不直观 |
| `read_all()` | 读取缓冲区全部数据 | 适合已知数据已到达的场景 |
| `in_waiting` | 缓冲区中待读字节数（属性） | 配合 `read(in_waiting)` 做非阻塞读取[citation:3] |

**编码问题**：`readline()` 返回 bytes，通常需要 `.decode('utf-8')` 转字符串。二进制数据用 `.hex()` 查看更安全[citation:3]。

**`readline()` 的坑**：如果超时发生时仍未遇到换行符，返回值**不包含** `\n`，需要通过是否含换行来判断是正常读完一行还是超时截断[citation:2]。

In [ ]:
# 使用 loop:// 虚拟串口演示读写（无需硬件）
import serial

with serial.serial_for_url('loop://', timeout=1) as ser:
    # 写入 bytes
    n = ser.write(b'Hello Serial\n')
    print(f"写入 {n} 字节")
    
    # 读回
    data = ser.read(13)          # 读 13 字节
    print("read:", data)
    
    # readline 演示
    ser.write(b'line1\nline2\n')
    print("readline:", ser.readline())   # b'line1\n'
    print("readline:", ser.readline())   # b'line2\n'

print("\n--- 字符串发送的易错点 ---")
# ser.write('hello')   # ❌ TypeError: 需要 bytes
# ser.write(b'hello')  # ✅
# ser.write('hello'.encode())  # ✅

## 4. 超时与缓冲区

**`timeout` 的语义**
- `timeout=None`（默认）：读操作**永久阻塞**，直到读到所需数据。
- `timeout=0`：**非阻塞**，立即返回缓冲区现有数据（可能为空）。
- `timeout=N`：最多等待 N 秒，超时后返回已读到的部分。

**缓冲区管理**

| 方法 | 作用 |
|---|---|
| `reset_input_buffer()` | 清空接收缓冲区，**发指令前建议调用**，避免读到旧数据[citation:3] |
| `reset_output_buffer()` | 清空发送缓冲区 |
| `flush()` | 等待所有数据发送完毕 |
| `in_waiting` | 接收缓冲区待读字节数 |

**非阻塞读取模式**：`timeout=0` 配合 `in_waiting` 轮询，适合需要同时处理其他逻辑的场景：
```python
if ser.in_waiting > 0:
    data = ser.read(ser.in_waiting)
```[citation:3]

In [ ]:
with serial.serial_for_url('loop://', timeout=0) as ser:
    ser.write(b'abc')
    print("in_waiting:", ser.in_waiting)       # 3
    
    # 非阻塞读取
    if ser.in_waiting > 0:
        data = ser.read(ser.in_waiting)
        print("读到:", data)
    
    # 清空缓冲区
    ser.write(b'xyz')
    ser.reset_input_buffer()
    print("清空后 in_waiting:", ser.in_waiting) # 0
    print("timeout=0 时 read 返回:", ser.read(10)) # b''

## 5. 异常处理

pyserial 定义了两个核心异常[citation:4]：

| 异常 | 触发场景 |
|---|---|
| `serial.SerialException` | 端口打开失败、设备断开、权限不足等**基类异常** |
| `serial.SerialTimeoutException` | 写超时（设置了 `write_timeout` 时） |

**标准处理模式**
```python
try:
    ser = serial.Serial('COM3', 9600, timeout=1)
except serial.SerialException as e:
    print(f"无法打开串口: {e}")
```

**常见运行时异常**
- `PermissionError`：Linux 上用户不在 `dialout` 组，或端口被其他程序占用[citation:3][citation:6]。
- `SerialException: could not open port`：端口名错误或设备未连接[citation:12]。
- `Device busy`：另一个进程已经打开了该端口。

In [ ]:
# 尝试打开一个不存在的端口
try:
    ser = serial.Serial('/dev/ttyNONEXIST', 9600, timeout=1)
except serial.SerialException as e:
    print(f"捕获 SerialException: {e}")

# 安全的读写封装模板
def safe_serial_demo(port, baud=9600):
    ser = None
    try:
        ser = serial.Serial(port, baud, timeout=1)
        ser.write(b'AT\r\n')
        response = ser.readline().decode('utf-8', errors='replace')
        return response
    except serial.SerialException as e:
        return f"串口错误: {e}"
    finally:
        if ser and ser.is_open:
            ser.close()

print(safe_serial_demo('/dev/ttyNONEXIST'))

## 6. 命令行工具

pyserial 自带两个实用的命令行工具，调试时非常方便[citation:1][citation:2]：

**① 列出端口**
```bash
python -m serial.tools.list_ports
python -m serial.tools.list_ports USB    # 按正则过滤
```

**② 简易终端 miniterm**
```bash
python -m serial.tools.miniterm COM3 115200
```
直接打开串口进行交互式收发，无需写代码，适合快速验证设备是否正常工作。`-h` 查看所有选项。

**关于 `readline()` 换行符**：如果需要自定义行尾或使用通用换行模式，推荐用 `io.TextIOWrapper` 包装，而不是依赖 `readline()` 的旧 `eol` 参数（新版已不再支持）[citation:1][citation:2]。

In [ ]:
# 用 io.TextIOWrapper 包装，获得文本行读写能力
import io

with serial.serial_for_url('loop://', timeout=1) as ser:
    sio = io.TextIOWrapper(io.BufferedRWPair(ser, ser))
    sio.write("hello\n")
    sio.flush()
    line = sio.readline()
    print("TextIOWrapper 读到:", repr(line))

## 7. 综合练习

1. 写一个函数 `list_serial_devices()`，打印所有可用串口的设备名和描述；若无设备则返回空列表。
2. 用 `loop://` 实现一个 echo 测试：写入 `b'PING\n'`，读取响应并解码打印。
3. 写一个带超时和异常保护的 `send_at_command(port, cmd)` 函数，发送 AT 指令并返回设备响应文本。

**参考答案**

In [ ]:
def list_serial_devices():
    devices = []
    for p in serial.tools.list_ports.comports():
        devices.append({"device": p.device, "desc": p.description})
    return devices

print("设备列表:", list_serial_devices())

# echo 测试
with serial.serial_for_url('loop://', timeout=1) as ser:
    ser.write(b'PING\n')
    echo = ser.readline().decode().strip()
    print("Echo:", echo)

# AT 指令封装
def send_at_command(port, cmd='AT\r\n', baud=9600):
    ser = None
    try:
        ser = serial.Serial(port, baud, timeout=1)
        ser.reset_input_buffer()
        ser.write(cmd.encode() if isinstance(cmd, str) else cmd)
        return ser.readline().decode('utf-8', errors='replace').strip()
    except serial.SerialException as e:
        return f"错误: {e}"
    finally:
        if ser and ser.is_open:
            ser.close()

print("AT 结果:", send_at_command('/dev/ttyNONEXIST'))

## 常见坑点总结

- **安装 `pyserial`，导入 `serial`**：`pip install pyserial`，代码写 `import serial`，不是 `import pyserial`[citation:12][citation:15]。
- **不设 `timeout` 会永久阻塞**：`read()` 和 `readline()` 在无数据时会一直等待，打开端口时务必设置 `timeout`[citation:1][citation:2]。
- **`write()` 只接受 bytes**：Python 3 中字符串先 `.encode()`，否则 TypeError[citation:3][citation:5]。
- **Linux 权限问题**：用户需加入 `dialout` 组，`sudo usermod -aG dialout $USER` 后重新登录生效[citation:3][citation:6]。
- **发指令前清空输入缓冲区**：用 `reset_input_buffer()` 避免读到上一次通信的残留数据[citation:3]。
- **`readline()` 超时截断**：超时时返回的内容不含 `\n`，与正常读完一行需区分处理[citation:2]。
- **端口被占用**：确保串口调试助手、Arduino IDE 等程序已关闭，同一端口不能被两个进程同时打开。
- **`loop://` 是学习利器**：无硬件时用它验证代码逻辑，有硬件后再替换为真实端口名。